# Funções Utilitárias Centralizadas (Squad 2)
**Projeto: Merca Data Platform**

> Este notebook centraliza todas as funções reutilizáveis do projeto, como conexões, leitura do Data Lake, manipulação de arquivos Delta nativos e controle de checkpoints de idempotência.

**Decisões Arquiteturais da Squad:**
* **Design Pattern (Shared Library):** O encapsulamento de funções garante o princípio DRY (*Don't Repeat Yourself*). Qualquer alteração de infraestrutura futura (ex: troca de chaves ou mudança de container) é feita apenas aqui, refletindo automaticamente em todas as camadas do pipeline (Bronze, Silver e Gold).
* **Padrão de Controle de Estado:** Conforme alinhamento técnico, todos os logs de controle de processamento de pacotes (`processed_snapshots`) utilizam estritamente o formato **`.json`**, garantindo maior flexibilidade, velocidade de leitura estruturada e padronização entre os desenvolvedores.

In [0]:
import os
import io
import time
import logging
import pandas as pd
import json
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# ─────────────────────────────────────────────
# CONFIGURAÇÃO DE LOGS
# ─────────────────────────────────────────────
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger("squad2")

# ─────────────────────────────────────────────
# CARREGAMENTO DE CREDENCIAIS
# ─────────────────────────────────────────────
load_dotenv()

# ADLS
ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")
SQUAD2_CONTAINER = os.getenv("SQUAD2_CONTAINER", "squad2")

# SQL Server
SQL_HOST     = os.getenv("SQL_HOST")
SQL_DATABASE = os.getenv("SQL_DATABASE")
SQL_USERNAME = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")

# Opções SQL reutilizáveis
SQL_OPTIONS = {
    "host"    : SQL_HOST,
    "database": SQL_DATABASE,
    "user"    : SQL_USERNAME,
    "password": SQL_PASSWORD
}

# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO PROJETO
# ─────────────────────────────────────────────
PATHS = {
    "raw"        : "real-time-data",
    "bronze"     : "squad2/bronze",
    "silver"     : "squad2/silver",
    "gold"       : "squad2/gold",
    "checkpoint" : "squad2/checkpoints"
}

TABELAS_SQUAD2 = [
    "ecommerce_categorias",
    "ecommerce_itens_pedido",
    "ecommerce_produtos",
    "ecommerce_pedidos"
]

SQL_SCHEMA = "squad2"
SQL_PREFIX = ""

# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO SPARK E DELTA LAKE
# ─────────────────────────────────────────────
try:
    # Habilita a governança física contínua e silenciosa (Serverless Friendly)
    spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
    spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    log.info("⚙️  Governança automática do Delta Lake ativada (AutoCompact & OptimizeWrite)")
except Exception as e:
    log.warning(f"Aviso: Não foi possível aplicar as configurações globais do Spark. ({e})")

# ─────────────────────────────────────────────
# FUNÇÕES — ADLS
# ─────────────────────────────────────────────
def get_adls_client() -> DataLakeServiceClient:
    credential = ClientSecretCredential(
        tenant_id     = ADLS_TENANT_ID,
        client_id     = ADLS_CLIENT_ID,
        client_secret = ADLS_CLIENT_SECRET
    )
    return DataLakeServiceClient(
        account_url = f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential  = credential
    )

def get_container_client():
    return get_adls_client().get_file_system_client(ADLS_CONTAINER)

def listar_snapshots(base_path: str = None) -> set:
    if base_path is None:
        base_path = PATHS["raw"]

    snapshots        = set()
    container_client = get_container_client()
    paths            = container_client.get_paths(
        path      = base_path,
        recursive = True
    )

    for item in paths:
        partes = item.name.replace(base_path + "/", "").split("/")
        if len(partes) == 4 and item.is_directory:
            snapshots.add("/".join(partes))

    return snapshots

def get_squad2_client():
    return get_adls_client().get_file_system_client(SQUAD2_CONTAINER)

def ler_parquet(snapshot_id: str, tabela: str) -> "pyspark.sql.DataFrame":
    base_path        = PATHS["raw"]
    file_path        = f"{base_path}/{snapshot_id}/{tabela}.parquet"
    container_client = get_container_client()
    file_client      = container_client.get_file_client(file_path)

    bytes_data = file_client.download_file().readall()
    pdf        = pd.read_parquet(io.BytesIO(bytes_data))

    return spark.createDataFrame(pdf)

# ─────────────────────────────────────────────
# FUNÇÕES — DELTA LAKE (ADLS via deltalake-python)
# ─────────────────────────────────────────────
def get_delta_path(camada: str, tabela: str) -> str:
    return (
        f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}"
        f".dfs.core.windows.net/{camada}/{tabela}"
    )

def get_storage_options() -> dict:
    return {
        "account_name"  : ADLS_STORAGE_ACCOUNT,
        "tenant_id"     : ADLS_TENANT_ID,
        "client_id"     : ADLS_CLIENT_ID,
        "client_secret" : ADLS_CLIENT_SECRET
    }

def delta_existe(camada: str, tabela: str) -> bool:
    try:
        from deltalake import DeltaTable
        DeltaTable(
            get_delta_path(camada, tabela),
            storage_options=get_storage_options()
        )
        return True
    except Exception:
        return False

def gravar_delta(
    df     : "pyspark.sql.DataFrame",
    camada : str,
    tabela : str,
    mode   : str = "append",
    partition_by : list = None
) -> bool:
    import pyarrow as pa
    from deltalake.writer import write_deltalake

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()
    modo_real    = mode if delta_existe(camada, tabela) else "overwrite"

    try:
        pdf          = df.toPandas()
        tabela_arrow = pa.Table.from_pandas(pdf)

        write_deltalake(
            table_or_uri    = path,
            data            = tabela_arrow,
            mode            = modo_real,
            storage_options = storage_opts,
            partition_by    = partition_by
        )

        log.info(f"Gravado: {path} → {len(pdf)} linhas | modo: {modo_real}")
        return True

    except Exception as e:
        log.error(f"Erro ao gravar {path}: {str(e)}")
        return False

def ler_delta(camada: str, tabela: str) -> "pyspark.sql.DataFrame":
    import io
    import pandas as pd

    path             = f"{camada}/{tabela}"
    squad2_client    = get_squad2_client()
    storage_opts     = get_storage_options()

    try:
        from deltalake import DeltaTable
        dt  = DeltaTable(
            get_delta_path(camada, tabela),
            storage_options=storage_opts
        )
        pdf = dt.to_pandas()
        log.info(f"Lido via deltalake: {len(pdf)} linhas")
        return spark.createDataFrame(pdf)

    except Exception as e1:
        log.warning(f"deltalake falhou ({str(e1)[:50]}), tentando via Azure SDK...")
        frames = []
        paths  = list(squad2_client.get_paths(
            path      = path,
            recursive = True
        ))

        parquets = [
            p.name for p in paths
            if p.name.endswith(".parquet")
            and "_delta_log" not in p.name
        ]

        for p in parquets:
            file_client = squad2_client.get_file_client(p)
            bytes_data  = file_client.download_file().readall()
            pdf         = pd.read_parquet(io.BytesIO(bytes_data))
            frames.append(pdf)

        if frames:
            pdf_total = pd.concat(frames, ignore_index=True)
            log.info(f"Lido via Azure SDK: {len(pdf_total)} linhas")
            return spark.createDataFrame(pdf_total)
        else:
            raise Exception(f"Nenhum arquivo parquet encontrado em {path}")

# ─────────────────────────────────────────────
# FUNÇÕES — CHECKPOINT
# ─────────────────────────────────────────────
def get_checkpoint(camada: str, tabela: str) -> tuple:
    # 1. Aponta para o cofre (squad2) e NÃO para o raw
    container_client = get_squad2_client()
    # 2. A estrutura sugerida pelo squad (Control ao nível das camadas)
    checkpoint_file  = f"control/{camada}/{tabela}/processed_snapshots.json"
    return container_client, checkpoint_file

def ler_checkpoint(camada: str, tabela: str) -> set:
    container_client, checkpoint_file = get_checkpoint(camada, tabela)
    processados                       = set()

    try:
        file_client = container_client.get_file_client(checkpoint_file)
        download    = file_client.download_file()
        conteudo    = download.readall().decode("utf-8")
        # Lê o JSON e converte a lista de volta para um conjunto (set)
        lista_processados = json.loads(conteudo)
        processados = set(lista_processados)
        log.info(f"{len(processados)} snapshot(s) já processado(s) lidos do JSON")
    except Exception:
        log.info("Nenhum checkpoint JSON encontrado")

    return processados

def salvar_checkpoint(camada: str, tabela: str, processados: set) -> None:
    container_client, checkpoint_file = get_checkpoint(camada, tabela)
    file_client = container_client.get_file_client(checkpoint_file)
    
    # Converte o conjunto (set) para lista e depois para JSON estruturado
    conteudo_json = json.dumps(list(processados), indent=4)
    
    try:
        # Tenta atualizar se o arquivo já existir
        file_client.get_file_properties()
        file_client.upload_data(conteudo_json, overwrite=True)
    except Exception:
        # Se não existir, cria o arquivo do zero
        file_client.create_file()
        file_client.upload_data(conteudo_json, overwrite=True)
        
    log.info(f"Checkpoint JSON salvo -> {len(processados)} snapshots")

# ─────────────────────────────────────────────
# FUNÇÕES — UTILITÁRIAS
# ─────────────────────────────────────────────
def log_inicio(notebook: str) -> datetime:
    inicio = datetime.now()
    log.info(f"{'='*50}")
    log.info(f"INÍCIO: {notebook}")
    log.info(f"Data  : {inicio.strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"{'='*50}")
    return inicio

def _validar_credenciais() -> None:
    credenciais = {
        "ADLS_CLIENT_ID"      : ADLS_CLIENT_ID,
        "ADLS_TENANT_ID"      : ADLS_TENANT_ID,
        "ADLS_CLIENT_SECRET"  : ADLS_CLIENT_SECRET,
        "ADLS_STORAGE_ACCOUNT": ADLS_STORAGE_ACCOUNT,
        "ADLS_CONTAINER"      : ADLS_CONTAINER,
    }
    todas_ok = True
    for nome, valor in credenciais.items():
        if not valor:
            log.error(f"Credencial não encontrada: {nome}")
            todas_ok = False

    if todas_ok:
        log.info("Helpers carregados! Todas as credenciais OK.")
    else:
        raise EnvironmentError("Credenciais ausentes. Verifique o arquivo .env")

_validar_credenciais()